# **Predict Stock Data**

## Install package

In [ ]:
%pip install matplotlib
%pip install sklearn
%pip install keras
%pip install tensorflow
%pip install plotly

In [ ]:
%pip install seaborn

## Step 1: Import Library

In [ ]:
import numpy as pd
import pandas as pd
import matplotlib.pyplot as plt

## Step 2: PreProcessing Data

### 2.1 Connect to S3 using Spark and Read Data 

#### *Start*

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

class spark_connection:
    # Create a SparkSession
    builder = SparkSession.builder.appName("Data Lakehouse") \
        .master('local[*]') \
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    # Get the SparkContext from the SparkSession
    sc = spark.sparkContext
    # Set the MinIO access key, secret key, endpoint, and other configurations
    sc._jsc.hadoopConfiguration().set("fs.s3a.access.key", "khanhlq10")
    sc._jsc.hadoopConfiguration().set("fs.s3a.secret.key", "khanhlq10")
    sc._jsc.hadoopConfiguration().set("fs.s3a.endpoint", "http://localhost:9000")
    sc._jsc.hadoopConfiguration().set("fs.s3a.path.style.access", "true")
    sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")
    sc._jsc.hadoopConfiguration().set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    sc._jsc.hadoopConfiguration().set("fs.s3a.connection.ssl.enabled", "false")

bucket = 'stock'

24/11/14 16:59:20 WARN Utils: Your hostname, Lams-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.0.17 instead (on interface en0)
24/11/14 16:59:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/lamquockhanh/VSCodeProjects/data-lake-house/venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lamquockhanh/.ivy2/cache
The jars for the packages stored in: /Users/lamquockhanh/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-17edc65c-3139-4f17-8c4f-bca9d8fc3976;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 211ms :: artifacts dl 10ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

#### *Stop*

In [2]:
spark_connection.spark.stop()

#### *Read*

In [2]:
from pyspark.sql.functions import desc

conn = spark_connection.spark

df = conn.read.option('header',True).load(f"s3a://{bucket}/gold/stock_price").sort(desc('ngay'),desc('version'))

df = df.selectExpr('ma as symbols','ngay as date','gia_dong_cua as close ', \
          'gia_mo_cua as open','gia_cao_nhat as max_price','gia_thap_nhat as min_price').toPandas()

df.shape[0]

24/11/14 17:07:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


5608

In [7]:
df_fpt = df[df['symbols'] == 'FPT'].set_index("date",inplace=True)

# df_fpt.set_index("date",inplace=True)
data = df[['close']]
data

,close
0,22900
1,28700
2,38150
3,14500
4,128000
...,...
5603,11900
5604,84900
5605,58600
5606,23800


#### *Visualize*

In [ ]:
from pyspark.sql.functions import desc,first
# import config.connection as cn

spark_conn = spark_connection.spark
bucket = 'stock'

def incremental_load_gold_layer():
    df_from_silver_layer = spark_conn.read.option('header',True).load(f"s3a://{bucket}/silver/stock_price").sort(desc('ngay'),desc('version'))

    # df_from_silver_layer.show()
    temp = df_from_silver_layer.orderBy(desc("version")).groupBy("ngay","ma").agg(first("version").alias("version"))
    # temp.sort(desc("ngay")).show()

    df_to_gold_layer = df_from_silver_layer.join(temp,['ma','ngay','version'])

    # df_to_gold_layer.orderBy(["ngay", "version"], ascending=[False, False]).show()
    df_to_gold_layer.orderBy(["ngay", "version"], ascending=[False, False]).write.format("delta").mode('append').save(f"s3a://{bucket}/gold/stock_price")
incremental_load_gold_layer()